# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all RecordSets and their properties
record_sets = metadata.record_sets()
print(f"Total Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field: {field.name:30} @id: {field.id:50} @type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect the @id of the main record set to analyze (here: the largest/first RecordSet)
record_sets = metadata.record_sets()
if len(record_sets) == 0:
    raise Exception('No RecordSets found in this Croissant dataset')

# For demonstration, pick the first record set
record_set_id = record_sets[0].id
print(f"Selected record set: {record_sets[0].name} (@id: {record_set_id})")

# List all record set ids for potential multi-table processing
record_set_ids = [rs.id for rs in record_sets]
# Load all tabular data into pd.DataFrame
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show columns for main record set
df_main = dataframes[record_set_id]
print(f"Columns in main record set: {df_main.columns.tolist()}")
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes outlier removal, data transformations, or grouping.

In [ ]:
# Inspect numeric columns and select one for analysis
numeric_candidates = df_main.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_candidates) == 0:
    # Try to convert any columns that look like numbers
    for col in df_main.columns:
        # Heuristic: try to convert, if many valid float entries, consider numeric
        try:
            series = pd.to_numeric(df_main[col], errors='coerce')
            valid_ratio = series.notna().mean()
            if valid_ratio > 0.5 and series.nunique() > 10:
                df_main[col] = series
                numeric_candidates.append(col)
        except:
            pass
if len(numeric_candidates) == 0:
    raise Exception("No numeric fields found for EDA.")
numeric_field = numeric_candidates[0]  # Use the first numeric column
print(f"Selected numeric field for EDA: {numeric_field}")

threshold = df_main[numeric_field].mean()  # Example: filter above mean
filtered_df = df_main[df_main[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a categorical column, e.g., 'Sample', 'Type', etc.
cat_candidates = df_main.select_dtypes(include=['object']).columns.tolist()
group_field = None
for col in cat_candidates:
    if df_main[col].nunique() < len(df_main) // 2:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical column found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(6,4))
sns.histplot(df_main[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouped field was found, show boxplot of numeric_field by group
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 protein mass spectrometry dataset using the `mlcroissant` library. We identified record sets, loaded tabular data, and performed basic EDA such as filtering, normalization, grouping, and visualized key numeric fields.

**Key takeaways:**
- The dataset includes detailed information on protein abundance and modifications from extracellular vesicles in human mast cells.
- The Croissant schema allows for flexible exploration and extraction by referencing entities via their `@id`.
- EDA steps provided a glimpse into the distributions and relationships in the data, suitable for downstream biomarker or comparative analysis tasks.

> You can further extend this notebook to perform more advanced analysis or machine learning using the `dataframes` dictionary for tabular access.